# HAKE-MER — Error analysis (+M1+M2, DistilBERT)

**Goal:** test-set forward pass on saved checkpoints; export confusion motifs and grief/relief/neutral examples.

**Protocol:** same clone and GPU setup as other campaign notebooks. If `runs/distilbert_base_uncased_seed*_m1_m2/best_model.pt` is missing, the notebook retrains M1+M2 (batch 16, LR 5e-5, 4 epochs, 3 seeds).

1. **Runtime → Factory reset** → **GPU**
2. Run all cells → download zip → archive in repo (see last cell)

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Runtime → Change runtime type → GPU")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import subprocess
from getpass import getpass
from pathlib import Path

REPO, WORKDIR = "khalef-khalil/marii", Path("/content/marii")
PUBLIC_URL = f"https://github.com/{REPO}.git"
TRAIN_FLAGS = "--epochs 4 --batch-size 16 --lr 5e-5 --early-stopping-patience 0"

def clone_repo() -> None:
    if WORKDIR.is_dir():
        subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(WORKDIR), "reset", "--hard", "origin/main"], check=True)
        return
    r = subprocess.run(["git", "clone", "--depth", "1", PUBLIC_URL, str(WORKDIR)], capture_output=True)
    if r.returncode == 0:
        return
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = getpass("GitHub token: ")
    subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/{REPO}.git", str(WORKDIR)], check=True)

clone_repo()
%cd {WORKDIR}
!git rev-parse --short HEAD

In [ ]:
!pip install -q -r requirements-train.txt

In [ ]:
from pathlib import Path

ckpts = sorted(Path("runs").glob("distilbert_base_uncased_seed*_m1_m2/best_model.pt"))
print(len(ckpts), "checkpoints found (need 3 for full error analysis)")
if len(ckpts) >= 3:
    print("Using existing checkpoints.")
else:
    print("Training M1+M2: 3 seeds × 4 epochs (epoch logs below; first line may take a few minutes)…")
    get_ipython().system(
        f"./run_m1_m2_campaign.sh --backbone distilbert-base-uncased {TRAIN_FLAGS}"
    )

In [ ]:
!./run_error_analysis_batch.sh

In [ ]:
import json
from pathlib import Path
agg = json.loads(Path("reference/artifacts/error_analysis_m1_m2_aggregate.json").read_text(encoding="utf-8"))
for row in agg["top_confusion_pairs_aggregate"][:8]:
    print(row["count"], row["pattern"])

In [ ]:
import zipfile
from google.colab import files
from pathlib import Path

out_name = "error_analysis_m1_m2.zip"
zip_path = Path(f"/content/{out_name}")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(Path("reference/artifacts").glob("error_analysis*.json")):
        zf.write(p, p.name)
    snippet = Path("error_analysis_snippet.tex")
    if snippet.is_file():
        zf.write(snippet, snippet.name)
print(f"Download {out_name} ({zip_path.stat().st_size / 1e3:.1f} KB)")
files.download(str(zip_path))

## Integrity

1. **File → Download → .ipynb** → `reference/training_records/step_error_analysis/colab/`
2. Archive zip + JSON (and snippet tex if present) under `reference/artifacts/`